In [ ]:

# ============================================================
# INSAT-3DS TIR1 — 15-MINUTE FRAME INTERPOLATION
# RIFE + Cloud Structure Attention Module
#      + Cloud Structure Gradient Consistency Loss
#
# Plot contains ONLY:
#   1. Input frame at T
#   2. Predicted frame at T+15 minutes
#   3. Input frame at T+30 minutes
#
# Saves:
#   - Predicted frames (.npy and .png)
#   - Three-panel plots (.png)
#   - Prediction timestamp CSV
#   - Failed frame CSV, if any
# ============================================================

import os
import re
import sys
import time
import h5py
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util

from pathlib import Path
from datetime import datetime, timedelta
from PIL import Image


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    "/home/kiruthigat.25mcse/Satellite_Frame_Interpolation"
)

DATA_DIR = PROJECT_DIR / "Dataset" / "Raw" / "GOES" / "INSAT"

MODEL_FILE = (
    PROJECT_DIR / "Research_RIFE" / "model"
    / "RIFE_CMAM_CMCL.py"
)

CHECKPOINT_PATH = (
    PROJECT_DIR / "checkpoints_NEW"
    / "RIFE_CMAM_CMCL_FROM_SCRATCH"
    / "epoch_100.pth"
)

OUTPUT_DIR = (
    PROJECT_DIR / "results"
    / "INSAT3DS_15MIN_ONLY_THREE_PLOTS"
)

PREDICTION_DIR = OUTPUT_DIR / "predicted_frames"
PLOT_DIR = OUTPUT_DIR / "three_frame_plots"

PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "15min_predictions.csv"
FAILED_CSV_PATH = OUTPUT_DIR / "failed_frames.csv"

DATASET_NAME = "IMG_TIR1"
PAD_MULTIPLE = 32

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("INSAT-3DS 15-MINUTE FRAME INTERPOLATION")
print("=" * 70)
print("Device     :", DEVICE)
print("Data       :", DATA_DIR)
print("Model      :", MODEL_FILE)
print("Checkpoint :", CHECKPOINT_PATH)
print("Output     :", OUTPUT_DIR)

for path in [DATA_DIR, MODEL_FILE, CHECKPOINT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Path not found: {path}")


# ============================================================
# 2. IMPORT CUSTOM MODEL
# ============================================================

sys.path.insert(0, str(PROJECT_DIR / "Research_RIFE"))

spec = importlib.util.spec_from_file_location(
    "rife_cmam_cmcl",
    str(MODEL_FILE)
)

if spec is None or spec.loader is None:
    raise ImportError(f"Cannot import model file: {MODEL_FILE}")

custom_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = custom_module
spec.loader.exec_module(custom_module)

if not hasattr(custom_module, "Model"):
    raise AttributeError(
        "A class named Model was not found in RIFE_CMAM_CMCL.py. "
        "Check the class name in your model file."
    )

# This may be a wrapper rather than torch.nn.Module.
network = custom_module.Model()

print("Custom Model wrapper created:", type(network))


# ============================================================
# 3. LOAD CHECKPOINT INTO THE CORRECT PYTORCH MODULE
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys())[:40])


def extract_state_dict(checkpoint_object):
    """Extract model weights from common checkpoint formats."""

    if not isinstance(checkpoint_object, dict):
        raise TypeError(
            "Checkpoint is not a dictionary. "
            "Please inspect its format."
        )

    possible_keys = [
        "model_state_dict",
        "state_dict",
        "model",
        "flownet",
        "network",
        "net",
        "weights"
    ]

    for key in possible_keys:
        value = checkpoint_object.get(key)

        if isinstance(value, dict) and value:
            if all(torch.is_tensor(v) for v in value.values()):
                print("Using weights stored under:", key)
                return value

    # Checkpoint may itself be a state dictionary.
    if checkpoint_object and all(
        torch.is_tensor(value)
        for value in checkpoint_object.values()
    ):
        print("Checkpoint itself is a state dictionary.")
        return checkpoint_object

    raise RuntimeError(
        "Could not identify the state dictionary. "
        "Available checkpoint keys: "
        + str(list(checkpoint_object.keys())[:60])
    )


state_dict = extract_state_dict(checkpoint)

# Remove common prefixes only when evaluating matching names below.
prefixes_to_remove = [
    "module.",
    "model.",
    "network.",
    "net.",
    "flownet.",
    "rife.",
    "IFNet."
]


def make_key_variants(original_state):
    """
    Generate plausible prefix-stripped state dictionaries.
    This does not modify tensor values.
    """

    variants = [("unchanged", original_state)]

    current = original_state
    removed = []

    while True:
        matching_prefix = None

        for prefix in prefixes_to_remove:
            if current and all(
                key.startswith(prefix) for key in current.keys()
            ):
                matching_prefix = prefix
                break

        if matching_prefix is None:
            break

        current = {
            key[len(matching_prefix):]: value
            for key, value in current.items()
        }
        removed.append(matching_prefix)

        variants.append((
            "removed " + "".join(removed),
            current
        ))

    # Try common single-prefix removals independently.
    for prefix in prefixes_to_remove:
        if state_dict and all(
            key.startswith(prefix) for key in state_dict.keys()
        ):
            variants.append((
                f"removed {prefix}",
                {
                    key[len(prefix):]: value
                    for key, value in state_dict.items()
                }
            ))

    # Remove duplicate variants.
    unique = []
    seen = set()

    for name, variant in variants:
        signature = tuple(variant.keys())

        if signature not in seen:
            seen.add(signature)
            unique.append((name, variant))

    return unique


# Find internal torch.nn.Module objects in the wrapper.
module_candidates = []

if isinstance(network, torch.nn.Module):
    module_candidates.append(("network", network))

for attr_name, attr_value in vars(network).items():
    if isinstance(attr_value, torch.nn.Module):
        module_candidates.append(
            (f"network.{attr_name}", attr_value)
        )

# Some wrappers hold nested modules.
for parent_name, parent_module in list(module_candidates):
    for child_name, child_module in parent_module.named_modules():
        if child_name and isinstance(child_module, torch.nn.Module):
            full_name = f"{parent_name}.{child_name}"
            module_candidates.append((full_name, child_module))

# Deduplicate references to the same module.
unique_candidates = []
seen_ids = set()

for name, module in module_candidates:
    if id(module) not in seen_ids:
        seen_ids.add(id(module))
        unique_candidates.append((name, module))

if not unique_candidates:
    print("\nWrapper attributes:")
    print([name for name in dir(network) if not name.startswith("__")])

    raise AttributeError(
        "No internal torch.nn.Module was found inside Model. "
        "The model wrapper must be inspected to determine how it "
        "loads its checkpoint."
    )

variants = make_key_variants(state_dict)

# Find an internal module whose parameter names and tensor shapes
# match a transformed version of the checkpoint.
best_match = None

for candidate_name, candidate_module in unique_candidates:
    target_state = candidate_module.state_dict()

    for variant_name, variant_state in variants:
        same_keys = set(target_state.keys()) == set(variant_state.keys())

        same_shapes = same_keys and all(
            target_state[key].shape == variant_state[key].shape
            for key in target_state.keys()
        )

        if same_shapes:
            best_match = (
                candidate_name,
                candidate_module,
                variant_name,
                variant_state
            )
            break

    if best_match is not None:
        break

if best_match is None:
    print("\nInternal PyTorch modules discovered:")

    for candidate_name, candidate_module in unique_candidates:
        target_state = candidate_module.state_dict()
        print(
            candidate_name,
            "| tensors:", len(target_state),
            "| sample keys:", list(target_state.keys())[:8]
        )

    print("\nCheckpoint sample keys:", list(state_dict.keys())[:20])

    raise RuntimeError(
        "Checkpoint names or tensor shapes do not exactly match "
        "any internal PyTorch module. The code has stopped to avoid "
        "running inference with incorrectly loaded weights. "
        "The Model class and checkpoint format must be matched."
    )

target_name, target_module, variant_name, matched_state = best_match

print("\nSelected internal module:", target_name)
print("Checkpoint key handling:", variant_name)

target_module.load_state_dict(
    matched_state,
    strict=True
)

# The wrapper itself is not a torch.nn.Module.
# Move the internal PyTorch network to the GPU.
network.flownet = network.flownet.to(DEVICE)
network.flownet.eval()

# Also move any other registered PyTorch modules, if needed.
for attr_name, attr_value in vars(network).items():
    if isinstance(attr_value, torch.nn.Module):
        attr_value.to(DEVICE)
        attr_value.eval()

print("Checkpoint loaded successfully.")
print("Internal network:", type(network.flownet))
print("Device:", next(network.flownet.parameters()).device)

print("Checkpoint loaded successfully with strict=True.")


# ============================================================
# 4. PARSE FRAME TIMESTAMPS
# ============================================================

def extract_timestamp(filepath):
    """
    Expected example:
    3SIMG_07JUL2026_0000_L1B_STD_V01R00_B3.h5
    """

    match = re.search(
        r"3SIMG_(\d{2}[A-Z]{3}\d{4})_(\d{4})",
        filepath.name.upper()
    )

    if match is None:
        return None

    try:
        return datetime.strptime(
            match.group(1) + "_" + match.group(2),
            "%d%b%Y_%H%M"
        )
    except ValueError:
        return None


all_files = sorted(DATA_DIR.rglob("*.h5"))

records = []

for filepath in all_files:
    timestamp = extract_timestamp(filepath)

    if timestamp is not None:
        records.append({
            "timestamp": timestamp,
            "path": filepath
        })

if not records:
    raise RuntimeError(
        "No HDF5 files matching the expected INSAT filename "
        f"pattern were found under {DATA_DIR}"
    )

records.sort(key=lambda item: item["timestamp"])

# Stop if multiple products share one timestamp.
timestamp_counts = {}

for record in records:
    timestamp_counts.setdefault(
        record["timestamp"], []
    ).append(record)

duplicates = {
    timestamp: items
    for timestamp, items in timestamp_counts.items()
    if len(items) > 1
}

if duplicates:
    example = [
        (str(timestamp), [str(item["path"]) for item in items])
        for timestamp, items in list(duplicates.items())[:5]
    ]

    raise RuntimeError(
        "Duplicate timestamps found. Filter DATA_DIR to the "
        "correct INSAT product before running.\n"
        + str(example)
    )

print("\nTimestamped HDF5 files:", len(records))
print("First timestamp:", records[0]["timestamp"])
print("Last timestamp :", records[-1]["timestamp"])


# ============================================================
# 5. LOAD IMG_TIR1
# ============================================================

def load_insat_frame(filepath):
    with h5py.File(filepath, "r") as h5:
        if DATASET_NAME not in h5:
            raise KeyError(
                f"Dataset {DATASET_NAME} not found in {filepath}"
            )

        frame = np.asarray(
            h5[DATASET_NAME][()]
        ).squeeze()

    if frame.ndim != 2:
        raise ValueError(
            f"Expected 2D image; got {frame.shape}: {filepath}"
        )

    frame = frame.astype(np.float32)

    if not np.isfinite(frame).any():
        raise ValueError(f"No finite pixels in {filepath}")

    return frame


# ============================================================
# 6. SHARED NORMALIZATION FOR THE TWO INPUTS
# ============================================================

def normalize_pair(frame0, frame1):
    valid0 = frame0[np.isfinite(frame0)]
    valid1 = frame1[np.isfinite(frame1)]

    combined = np.concatenate([valid0, valid1])

    low, high = np.percentile(
        combined, [1, 99]
    )

    if not np.isfinite(low) or not np.isfinite(high):
        raise ValueError("Invalid normalization limits.")

    if high <= low:
        high = low + 1e-6

    def normalize(frame):
        result = (frame - low) / (high - low)
        result = np.clip(result, 0.0, 1.0)
        result[~np.isfinite(frame)] = 0.0
        return result.astype(np.float32)

    return normalize(frame0), normalize(frame1), low, high


# ============================================================
# 7. CONVERT GRAYSCALE FRAME TO MODEL INPUT
# ============================================================

def prepare_tensor(image_2d):
    image_2d = np.nan_to_num(
        image_2d,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    image_3ch = np.repeat(
        image_2d[None, :, :],
        3,
        axis=0
    )

    tensor = torch.from_numpy(
        image_3ch
    ).float().unsqueeze(0).to(DEVICE)

    _, _, height, width = tensor.shape

    pad_h = (PAD_MULTIPLE - height % PAD_MULTIPLE) % PAD_MULTIPLE
    pad_w = (PAD_MULTIPLE - width % PAD_MULTIPLE) % PAD_MULTIPLE

    if pad_h or pad_w:
        tensor = torch.nn.functional.pad(
            tensor,
            (0, pad_w, 0, pad_h),
            mode="replicate"
        )

    return tensor, height, width


# ============================================================
# 8. MODEL INFERENCE
# ============================================================

def extract_prediction(output):
    if torch.is_tensor(output):
        return output

    if isinstance(output, (tuple, list)):
        for item in reversed(output):
            if torch.is_tensor(item):
                return item

    if isinstance(output, dict):
        for key in [
            "prediction", "output", "merged", "frame", "image"
        ]:
            if key in output:
                return extract_prediction(output[key])

        for value in output.values():
            if torch.is_tensor(value):
                return value

    raise TypeError(
        "Could not find a prediction tensor in the model output."
    )


@torch.inference_mode()
def predict_midpoint(frame0, frame1):
    img0, height, width = prepare_tensor(frame0)
    img1, _, _ = prepare_tensor(frame1)

    if not hasattr(network, "inference"):
        raise AttributeError(
            "The wrapper does not expose an inference() method. "
            "Check the Model class in RIFE_CMAM_CMCL.py."
        )

    # Support common RIFE wrapper inference signatures.
    try:
        output = network.inference(
            img0, img1, timestep=0.5
        )
    except TypeError:
        output = network.inference(img0, img1)

    prediction = extract_prediction(output)

    prediction = prediction[:, :, :height, :width]

    if prediction.ndim != 4:
        raise ValueError(
            f"Unexpected prediction shape: {prediction.shape}"
        )

    # Convert repeated RGB output to a single grayscale image.
    prediction = prediction[0].mean(dim=0)
    prediction = prediction.detach().float().cpu().numpy()

    if not np.isfinite(prediction).any():
        raise ValueError("Prediction contains no finite pixels.")

    prediction = np.nan_to_num(
        prediction,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    return np.clip(prediction, 0.0, 1.0)


# ============================================================
# 9. PLOT ONLY THE THREE REQUESTED FRAMES
# ============================================================

def save_three_panel_plot(
    frame0,
    predicted_frame,
    frame1,
    timestamp0,
    timestamp_pred,
    timestamp1,
    output_path
):
    fig, axes = plt.subplots(
        1, 3,
        figsize=(18, 6),
        constrained_layout=True
    )

    panels = [
        (
            frame0,
            f"Input Frame 1\n{timestamp0:%d-%b-%Y %H:%M}"
        ),
        (
            predicted_frame,
            f"Predicted Intermediate Frame\n"
            f"{timestamp_pred:%d-%b-%Y %H:%M}"
        ),
        (
            frame1,
            f"Input Frame 2\n{timestamp1:%d-%b-%Y %H:%M}"
        )
    ]

    for ax, (image, title) in zip(axes, panels):
        ax.imshow(
            image,
            cmap="gray",
            vmin=0,
            vmax=1,
            interpolation="nearest"
        )
        ax.set_title(title, fontsize=12)
        ax.axis("off")

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# 10. IDENTIFY CONSECUTIVE 30-MINUTE PAIRS
# ============================================================

pairs = []

for index in range(len(records) - 1):
    record0 = records[index]
    record1 = records[index + 1]

    if (
        record1["timestamp"] - record0["timestamp"]
        == timedelta(minutes=30)
    ):
        pairs.append((record0, record1))

print("\nConsecutive 30-minute pairs:", len(pairs))

if not pairs:
    raise RuntimeError(
        "No consecutive 30-minute pairs found. "
        "Check file timestamps and dataset coverage."
    )


# ============================================================
# 11. PREDICT AND SAVE ALL INTERMEDIATE FRAMES
# ============================================================

results = []
failed = []

start_time = time.time()

for index, (record0, record1) in enumerate(pairs, start=1):
    timestamp0 = record0["timestamp"]
    timestamp1 = record1["timestamp"]
    timestamp_pred = timestamp0 + timedelta(minutes=15)

    path0 = record0["path"]
    path1 = record1["path"]

    print(
        f"\n[{index}/{len(pairs)}] "
        f"{timestamp0:%Y-%m-%d %H:%M} -> "
        f"{timestamp_pred:%Y-%m-%d %H:%M} -> "
        f"{timestamp1:%Y-%m-%d %H:%M}"
    )

    try:
        frame0_raw = load_insat_frame(path0)
        frame1_raw = load_insat_frame(path1)

        if frame0_raw.shape != frame1_raw.shape:
            raise ValueError(
                f"Input dimensions differ: "
                f"{frame0_raw.shape} vs {frame1_raw.shape}"
            )

        frame0, frame1, norm_low, norm_high = normalize_pair(
            frame0_raw, frame1_raw
        )

        predicted_frame = predict_midpoint(
            frame0, frame1
        )

        time_label = timestamp_pred.strftime("%Y%m%d_%H%M")

        prediction_png = (
            PREDICTION_DIR
            / f"INSAT3DS_TIR1_PRED_{time_label}.png"
        )

        prediction_npy = (
            PREDICTION_DIR
            / f"INSAT3DS_TIR1_PRED_{time_label}.npy"
        )

        plot_path = (
            PLOT_DIR
            / f"INSAT3DS_15MIN_{time_label}.png"
        )

        # Save normalized prediction as an image.
        prediction_uint16 = np.round(
            predicted_frame * 65535
        ).astype(np.uint16)

        Image.fromarray(prediction_uint16).save(
            prediction_png
        )

        # Save floating-point normalized prediction.
        np.save(
            prediction_npy,
            predicted_frame.astype(np.float32)
        )

        # Create ONLY the three-panel plot.
        save_three_panel_plot(
            frame0,
            predicted_frame,
            frame1,
            timestamp0,
            timestamp_pred,
            timestamp1,
            plot_path
        )

        results.append({
            "input_frame_1": str(path0),
            "input_timestamp": timestamp0.isoformat(),
            "predicted_timestamp": timestamp_pred.isoformat(),
            "input_frame_2": str(path1),
            "input_2_timestamp": timestamp1.isoformat(),
            "prediction_png": str(prediction_png),
            "prediction_npy": str(prediction_npy),
            "three_panel_plot": str(plot_path),
            "height": int(predicted_frame.shape[0]),
            "width": int(predicted_frame.shape[1]),
            "normalization_low": float(norm_low),
            "normalization_high": float(norm_high)
        })

        print("Saved prediction:", prediction_png.name)
        print("Saved three-panel plot:", plot_path.name)

    except Exception as exc:
        print("FAILED:", repr(exc))

        failed.append({
            "input_frame_1": str(path0),
            "input_frame_2": str(path1),
            "error": repr(exc)
        })

    # Save progress during processing.
    pd.DataFrame(results).to_csv(
        CSV_PATH,
        index=False
    )

    if failed:
        pd.DataFrame(failed).to_csv(
            FAILED_CSV_PATH,
            index=False
        )


# ============================================================
# 12. FINAL SUMMARY
# ============================================================

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("PROCESSING FINISHED")
print("=" * 70)
print("Valid 30-minute pairs :", len(pairs))
print("Successful predictions:", len(results))
print("Failed pairs          :", len(failed))
print(f"Elapsed time          : {elapsed / 60:.2f} minutes")
print("Predicted frames      :", PREDICTION_DIR)
print("Three-panel plots     :", PLOT_DIR)
print("Prediction CSV        :", CSV_PATH)

if failed:
    print("Failed-pairs CSV      :", FAILED_CSV_PATH)

if results:
    print("\nExample plot:")
    print(results[0]["three_panel_plot"])


INSAT-3DS 15-MINUTE FRAME INTERPOLATION
Device     : cuda
Data       : /home/kiruthigat.25mcse/Satellite_Frame_Interpolation/Dataset/Raw/GOES/INSAT
Model      : /home/kiruthigat.25mcse/Satellite_Frame_Interpolation/Research_RIFE/model/RIFE_CMAM_CMCL.py
Checkpoint : /home/kiruthigat.25mcse/Satellite_Frame_Interpolation/checkpoints_NEW/RIFE_CMAM_CMCL_FROM_SCRATCH/epoch_100.pth
Output     : /home/kiruthigat.25mcse/Satellite_Frame_Interpolation/results/INSAT3DS_15MIN_ONLY_THREE_PLOTS
Custom Model wrapper created: <class 'rife_cmam_cmcl.Model'>

Checkpoint type: <class 'dict'>
Checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_states', 'metrics', 'learning_rate', 'cmcl_weight', 'distill_weight']
Using weights stored under: model_state_dict

Selected internal module: network.flownet
Checkpoint key handling: unchanged
Checkpoint loaded successfully.
Internal network: <class 'model.IFNet.IFNet'>
Device: cuda:0
Checkpoint loaded successfully with strict=True.

Timestamped HDF5 files: 426

In [3]:

# ============================================================
# 4. CREATE TIMESTAMPED INSAT RECORDS AND 30-MINUTE PAIRS
# ============================================================

def extract_timestamp(filepath):
    match = re.search(
        r"3SIMG_(\d{2}[A-Z]{3}\d{4})_(\d{4})",
        filepath.name.upper()
    )

    if match is None:
        return None

    try:
        return datetime.strptime(
            match.group(1) + "_" + match.group(2),
            "%d%b%Y_%H%M"
        )
    except ValueError:
        return None


all_files = sorted(DATA_DIR.rglob("*.h5"))

records = []

for filepath in all_files:
    timestamp = extract_timestamp(filepath)

    if timestamp is not None:
        records.append({
            "timestamp": timestamp,
            "path": filepath
        })

records.sort(key=lambda item: item["timestamp"])

if not records:
    raise RuntimeError("No valid INSAT HDF5 files found.")

timestamp_counts = {}

for record in records:
    timestamp_counts.setdefault(
        record["timestamp"], []
    ).append(record)

duplicates = {
    timestamp: items
    for timestamp, items in timestamp_counts.items()
    if len(items) > 1
}

if duplicates:
    raise RuntimeError(
        "Duplicate timestamps found. Check the INSAT dataset."
    )

pairs = []

for index in range(len(records) - 1):
    record0 = records[index]
    record1 = records[index + 1]

    if (
        record1["timestamp"] - record0["timestamp"]
        == timedelta(minutes=30)
    ):
        pairs.append((record0, record1))

print("Total INSAT HDF5 files:", len(records))
print("Total 30-minute pairs:", len(pairs))

if len(pairs) < 329:
    raise RuntimeError(
        f"Only {len(pairs)} pairs found. "
        "Pair 329 is not available."
    )

print("First pair:", pairs[0][0]["timestamp"])
print("Last pair :", pairs[-1][1]["timestamp"])


# ============================================================
# 5. LOAD INSAT IMG_TIR1
# ============================================================

def load_insat_frame(filepath):
    with h5py.File(filepath, "r") as h5:
        if DATASET_NAME not in h5:
            raise KeyError(
                f"{DATASET_NAME} not found in {filepath}"
            )

        frame = np.asarray(
            h5[DATASET_NAME][()]
        ).squeeze()

    if frame.ndim != 2:
        raise ValueError(
            f"Invalid image dimensions: {frame.shape}"
        )

    frame = frame.astype(np.float32)

    if not np.isfinite(frame).any():
        raise ValueError("No valid pixels found.")

    return frame


# ============================================================
# 6. NORMALIZATION
# ============================================================

def normalize_pair(frame0, frame1):
    valid0 = frame0[np.isfinite(frame0)]
    valid1 = frame1[np.isfinite(frame1)]

    combined = np.concatenate([valid0, valid1])

    low, high = np.percentile(
        combined, [1, 99]
    )

    if high <= low:
        high = low + 1e-6

    def normalize(frame):
        result = (frame - low) / (high - low)
        result = np.clip(result, 0, 1)
        result[~np.isfinite(frame)] = 0

        return result.astype(np.float32)

    return (
        normalize(frame0),
        normalize(frame1),
        low,
        high
    )


# ============================================================
# 7. PREPARE MODEL INPUT
# ============================================================

def prepare_tensor(image_2d):
    image_2d = np.nan_to_num(
        image_2d,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    image_3ch = np.repeat(
        image_2d[None, :, :],
        3,
        axis=0
    )

    tensor = torch.from_numpy(
        image_3ch
    ).float().unsqueeze(0).to(DEVICE)

    _, _, height, width = tensor.shape

    pad_h = (PAD_MULTIPLE - height % PAD_MULTIPLE) % PAD_MULTIPLE
    pad_w = (PAD_MULTIPLE - width % PAD_MULTIPLE) % PAD_MULTIPLE

    if pad_h or pad_w:
        tensor = torch.nn.functional.pad(
            tensor,
            (0, pad_w, 0, pad_h),
            mode="replicate"
        )

    return tensor, height, width


# ============================================================
# 8. PREDICTION
# ============================================================

def extract_prediction(output):
    if torch.is_tensor(output):
        return output

    if isinstance(output, (tuple, list)):
        for item in reversed(output):
            if torch.is_tensor(item):
                return item

    if isinstance(output, dict):
        for key in [
            "prediction", "output", "merged", "frame", "image"
        ]:
            if key in output:
                return extract_prediction(output[key])

        for value in output.values():
            if torch.is_tensor(value):
                return value

    raise TypeError("Prediction tensor not found.")


@torch.inference_mode()
def predict_midpoint(frame0, frame1):
    img0, height, width = prepare_tensor(frame0)
    img1, _, _ = prepare_tensor(frame1)

    output = network.inference(
        img0, img1, timestep=0.5
    )

    prediction = extract_prediction(output)

    prediction = prediction[:, :, :height, :width]

    prediction = prediction[0].mean(dim=0)
    prediction = prediction.detach().float().cpu().numpy()

    prediction = np.nan_to_num(
        prediction,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    return np.clip(prediction, 0, 1)


# ============================================================
# 9. SAVE THREE-PANEL PLOT
# ============================================================

def save_three_panel_plot(
    frame0,
    predicted_frame,
    frame1,
    timestamp0,
    timestamp_pred,
    timestamp1,
    output_path
):
    fig, axes = plt.subplots(
        1, 3,
        figsize=(18, 6),
        constrained_layout=True
    )

    panels = [
        (frame0, f"Input Frame 1\n{timestamp0:%d-%b-%Y %H:%M}"),
        (
            predicted_frame,
            f"Predicted Intermediate Frame\n"
            f"{timestamp_pred:%d-%b-%Y %H:%M}"
        ),
        (frame1, f"Input Frame 2\n{timestamp1:%d-%b-%Y %H:%M}")
    ]

    for ax, (image, title) in zip(axes, panels):
        ax.imshow(
            image,
            cmap="gray",
            vmin=0,
            vmax=1
        )
        ax.set_title(title, fontsize=12)
        ax.axis("off")

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(fig)


Total INSAT HDF5 files: 426
Total 30-minute pairs: 421
First pair: 2026-07-07 00:00:00
Last pair : 2026-07-15 23:30:00


In [4]:

import os
import re
import sys
import time
import h5py
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util

from pathlib import Path
from datetime import datetime, timedelta
from PIL import Image


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_DIR = Path(
    "/home/kiruthigat.25mcse/Satellite_Frame_Interpolation"
)

DATA_DIR = PROJECT_DIR / "Dataset" / "Raw" / "GOES" / "INSAT"

MODEL_FILE = (
    PROJECT_DIR / "Research_RIFE" / "model"
    / "RIFE_CMAM_CMCL.py"
)

CHECKPOINT_PATH = (
    PROJECT_DIR / "checkpoints_NEW"
    / "RIFE_CMAM_CMCL_FROM_SCRATCH"
    / "epoch_100.pth"
)

OUTPUT_DIR = (
    PROJECT_DIR / "results"
    / "INSAT3DS_15MIN_ONLY_THREE_PLOTS"
)

PREDICTION_DIR = OUTPUT_DIR / "predicted_frames"
PLOT_DIR = OUTPUT_DIR / "three_frame_plots"

PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "15min_predictions.csv"
FAILED_CSV_PATH = OUTPUT_DIR / "failed_frames.csv"

DATASET_NAME = "IMG_TIR1"
PAD_MULTIPLE = 32

START_PAIR = 329

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("INSAT-3DS 15-MINUTE FRAME INTERPOLATION")
print("=" * 70)

print("Device     :", DEVICE)
print("Data       :", DATA_DIR)
print("Model      :", MODEL_FILE)
print("Checkpoint :", CHECKPOINT_PATH)
print("Output     :", OUTPUT_DIR)

for path in [DATA_DIR, MODEL_FILE, CHECKPOINT_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Path not found: {path}")


# ============================================================
# 2. IMPORT CUSTOM RIFE MODEL
# ============================================================

sys.path.insert(
    0,
    str(PROJECT_DIR / "Research_RIFE")
)

spec = importlib.util.spec_from_file_location(
    "rife_cmam_cmcl",
    str(MODEL_FILE)
)

if spec is None or spec.loader is None:
    raise ImportError(
        f"Cannot import model: {MODEL_FILE}"
    )

custom_module = importlib.util.module_from_spec(spec)

sys.modules[spec.name] = custom_module

spec.loader.exec_module(custom_module)

if not hasattr(custom_module, "Model"):
    raise AttributeError(
        "Model class not found in RIFE_CMAM_CMCL.py"
    )

network = custom_module.Model()

print("Model wrapper:", type(network))


# ============================================================
# 3. LOAD CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print(
        "Checkpoint keys:",
        list(checkpoint.keys())[:40]
    )


def extract_state_dict(checkpoint_object):

    if not isinstance(checkpoint_object, dict):
        raise TypeError(
            "Checkpoint is not a dictionary."
        )

    possible_keys = [
        "model_state_dict",
        "state_dict",
        "model",
        "flownet",
        "network",
        "net",
        "weights"
    ]

    for key in possible_keys:

        value = checkpoint_object.get(key)

        if isinstance(value, dict) and value:

            if all(
                torch.is_tensor(v)
                for v in value.values()
            ):
                print(
                    "Using checkpoint weights:",
                    key
                )

                return value

    if checkpoint_object and all(
        torch.is_tensor(value)
        for value in checkpoint_object.values()
    ):

        print(
            "Checkpoint itself is a state dictionary."
        )

        return checkpoint_object

    raise RuntimeError(
        "Could not identify checkpoint state dictionary. "
        f"Available keys: {list(checkpoint_object.keys())[:60]}"
    )


state_dict = extract_state_dict(checkpoint)


# ============================================================
# 4. CHECKPOINT KEY VARIANTS
# ============================================================

prefixes_to_remove = [
    "module.",
    "model.",
    "network.",
    "net.",
    "flownet.",
    "rife.",
    "IFNet."
]


def make_key_variants(original_state):

    variants = [
        ("unchanged", original_state)
    ]

    current = original_state
    removed = []

    while True:

        matching_prefix = None

        for prefix in prefixes_to_remove:

            if current and all(
                key.startswith(prefix)
                for key in current.keys()
            ):

                matching_prefix = prefix
                break

        if matching_prefix is None:
            break

        current = {
            key[len(matching_prefix):]: value
            for key, value in current.items()
        }

        removed.append(matching_prefix)

        variants.append(
            (
                "removed " + "".join(removed),
                current
            )
        )

    for prefix in prefixes_to_remove:

        if state_dict and all(
            key.startswith(prefix)
            for key in state_dict.keys()
        ):

            variants.append(
                (
                    f"removed {prefix}",
                    {
                        key[len(prefix):]: value
                        for key, value in state_dict.items()
                    }
                )
            )

    unique = []
    seen = set()

    for name, variant in variants:

        signature = tuple(variant.keys())

        if signature not in seen:

            seen.add(signature)

            unique.append(
                (name, variant)
            )

    return unique


variants = make_key_variants(state_dict)


# ============================================================
# 5. FIND INTERNAL PYTORCH MODEL
# ============================================================

module_candidates = []

if isinstance(network, torch.nn.Module):

    module_candidates.append(
        ("network", network)
    )

for attr_name, attr_value in vars(network).items():

    if isinstance(attr_value, torch.nn.Module):

        module_candidates.append(
            (
                f"network.{attr_name}",
                attr_value
            )
        )

for parent_name, parent_module in list(module_candidates):

    for child_name, child_module in parent_module.named_modules():

        if child_name and isinstance(
            child_module,
            torch.nn.Module
        ):

            module_candidates.append(
                (
                    f"{parent_name}.{child_name}",
                    child_module
                )
            )

unique_candidates = []
seen_ids = set()

for name, module in module_candidates:

    if id(module) not in seen_ids:

        seen_ids.add(id(module))

        unique_candidates.append(
            (name, module)
        )

if not unique_candidates:

    raise AttributeError(
        "No internal torch.nn.Module found in Model wrapper."
    )


# ============================================================
# 6. MATCH CHECKPOINT AND LOAD STRICTLY
# ============================================================

best_match = None

for candidate_name, candidate_module in unique_candidates:

    target_state = candidate_module.state_dict()

    for variant_name, variant_state in variants:

        same_keys = (
            set(target_state.keys())
            == set(variant_state.keys())
        )

        same_shapes = same_keys and all(
            target_state[key].shape
            == variant_state[key].shape
            for key in target_state.keys()
        )

        if same_shapes:

            best_match = (
                candidate_name,
                candidate_module,
                variant_name,
                variant_state
            )

            break

    if best_match is not None:
        break


if best_match is None:

    print("\nAvailable internal modules:")

    for candidate_name, candidate_module in unique_candidates:

        target_state = candidate_module.state_dict()

        print(
            candidate_name,
            "Tensors:",
            len(target_state),
            "Sample keys:",
            list(target_state.keys())[:8]
        )

    print(
        "\nCheckpoint sample keys:",
        list(state_dict.keys())[:20]
    )

    raise RuntimeError(
        "Checkpoint tensor names or shapes do not match "
        "the model. Inference stopped to avoid using "
        "incorrect weights."
    )


target_name, target_module, variant_name, matched_state = best_match

print("\nSelected internal module:", target_name)
print("Checkpoint key handling:", variant_name)

target_module.load_state_dict(
    matched_state,
    strict=True
)

target_module.to(DEVICE)
target_module.eval()

if hasattr(network, "flownet"):

    network.flownet.to(DEVICE)
    network.flownet.eval()

for attr_name, attr_value in vars(network).items():

    if isinstance(attr_value, torch.nn.Module):

        attr_value.to(DEVICE)
        attr_value.eval()

print("\nCheckpoint loaded successfully.")
print("Strict loading: True")
print("Device:", DEVICE)


# ============================================================
# 7. EXTRACT INSAT TIMESTAMPS
# ============================================================

def extract_timestamp(filepath):

    match = re.search(
        r"3SIMG_(\d{2}[A-Z]{3}\d{4})_(\d{4})",
        filepath.name.upper()
    )

    if match is None:
        return None

    try:

        return datetime.strptime(
            match.group(1) + "_" + match.group(2),
            "%d%b%Y_%H%M"
        )

    except ValueError:

        return None


all_files = sorted(
    DATA_DIR.rglob("*.h5")
)

records = []

for filepath in all_files:

    timestamp = extract_timestamp(filepath)

    if timestamp is not None:

        records.append(
            {
                "timestamp": timestamp,
                "path": filepath
            }
        )

records.sort(
    key=lambda item: item["timestamp"]
)

if not records:

    raise RuntimeError(
        "No valid INSAT HDF5 files found."
    )


# ============================================================
# 8. CHECK DUPLICATE TIMESTAMPS
# ============================================================

timestamp_counts = {}

for record in records:

    timestamp_counts.setdefault(
        record["timestamp"],
        []
    ).append(record)


duplicates = {
    timestamp: items
    for timestamp, items in timestamp_counts.items()
    if len(items) > 1
}

if duplicates:

    example = [
        (
            str(timestamp),
            [str(item["path"]) for item in items]
        )
        for timestamp, items in list(duplicates.items())[:5]
    ]

    raise RuntimeError(
        "Duplicate timestamps found. "
        "Check the INSAT dataset.\n"
        + str(example)
    )

print("\nTimestamped INSAT files:", len(records))
print("First timestamp:", records[0]["timestamp"])
print("Last timestamp :", records[-1]["timestamp"])


# ============================================================
# 9. CREATE CONSECUTIVE 30-MINUTE PAIRS
# ============================================================

pairs = []

for index in range(len(records) - 1):

    record0 = records[index]
    record1 = records[index + 1]

    if (
        record1["timestamp"] - record0["timestamp"]
        == timedelta(minutes=30)
    ):

        pairs.append(
            (record0, record1)
        )

print("\nTotal 30-minute pairs:", len(pairs))

if not pairs:

    raise RuntimeError(
        "No consecutive 30-minute pairs found."
    )

if len(pairs) < START_PAIR:

    raise RuntimeError(
        f"Only {len(pairs)} pairs found. "
        f"Cannot start from pair {START_PAIR}."
    )


# ============================================================
# 10. LOAD IMG_TIR1
# ============================================================

def load_insat_frame(filepath):

    with h5py.File(filepath, "r") as h5:

        if DATASET_NAME not in h5:

            raise KeyError(
                f"{DATASET_NAME} not found in {filepath}"
            )

        frame = np.asarray(
            h5[DATASET_NAME][()]
        ).squeeze()

    if frame.ndim != 2:

        raise ValueError(
            f"Expected 2D image, got {frame.shape}"
        )

    frame = frame.astype(np.float32)

    if not np.isfinite(frame).any():

        raise ValueError(
            f"No finite pixels found in {filepath}"
        )

    return frame


# ============================================================
# 11. SHARED NORMALIZATION
# ============================================================

def normalize_pair(frame0, frame1):

    valid0 = frame0[np.isfinite(frame0)]
    valid1 = frame1[np.isfinite(frame1)]

    combined = np.concatenate(
        [valid0, valid1]
    )

    low, high = np.percentile(
        combined,
        [1, 99]
    )

    if not np.isfinite(low) or not np.isfinite(high):

        raise ValueError(
            "Invalid normalization limits."
        )

    if high <= low:

        high = low + 1e-6

    def normalize(frame):

        result = (
            frame - low
        ) / (
            high - low
        )

        result = np.clip(
            result,
            0.0,
            1.0
        )

        result[~np.isfinite(frame)] = 0.0

        return result.astype(np.float32)

    return (
        normalize(frame0),
        normalize(frame1),
        low,
        high
    )


# ============================================================
# 12. PREPARE MODEL INPUT
# ============================================================

def prepare_tensor(image_2d):

    image_2d = np.nan_to_num(
        image_2d,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    image_3ch = np.repeat(
        image_2d[None, :, :],
        3,
        axis=0
    )

    tensor = torch.from_numpy(
        image_3ch
    ).float().unsqueeze(0).to(DEVICE)

    _, _, height, width = tensor.shape

    pad_h = (
        PAD_MULTIPLE - height % PAD_MULTIPLE
    ) % PAD_MULTIPLE

    pad_w = (
        PAD_MULTIPLE - width % PAD_MULTIPLE
    ) % PAD_MULTIPLE

    if pad_h or pad_w:

        tensor = torch.nn.functional.pad(
            tensor,
            (0, pad_w, 0, pad_h),
            mode="replicate"
        )

    return tensor, height, width


# ============================================================
# 13. EXTRACT MODEL PREDICTION
# ============================================================

def extract_prediction(output):

    if torch.is_tensor(output):

        return output

    if isinstance(output, (tuple, list)):

        for item in reversed(output):

            if torch.is_tensor(item):

                return item

    if isinstance(output, dict):

        for key in [
            "prediction",
            "output",
            "merged",
            "frame",
            "image"
        ]:

            if key in output:

                return extract_prediction(
                    output[key]
                )

        for value in output.values():

            if torch.is_tensor(value):

                return value

    raise TypeError(
        "Could not find prediction tensor."
    )


# ============================================================
# 14. RIFE MIDPOINT INFERENCE
# ============================================================

@torch.inference_mode()
def predict_midpoint(frame0, frame1):

    img0, height, width = prepare_tensor(frame0)

    img1, _, _ = prepare_tensor(frame1)

    if not hasattr(network, "inference"):

        raise AttributeError(
            "Model wrapper does not contain inference()."
        )

    try:

        output = network.inference(
            img0,
            img1,
            timestep=0.5
        )

    except TypeError:

        output = network.inference(
            img0,
            img1
        )

    prediction = extract_prediction(
        output
    )

    if prediction.ndim != 4:

        raise ValueError(
            f"Unexpected prediction shape: {prediction.shape}"
        )

    prediction = prediction[:, :, :height, :width]

    prediction = prediction[0].mean(dim=0)

    prediction = (
        prediction.detach()
        .float()
        .cpu()
        .numpy()
    )

    if not np.isfinite(prediction).any():

        raise ValueError(
            "Prediction contains no finite pixels."
        )

    prediction = np.nan_to_num(
        prediction,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    return np.clip(
        prediction,
        0.0,
        1.0
    )


# ============================================================
# 15. SAVE THREE-PANEL VISUALIZATION
# ============================================================

def save_three_panel_plot(
    frame0,
    predicted_frame,
    frame1,
    timestamp0,
    timestamp_pred,
    timestamp1,
    output_path
):

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 6),
        constrained_layout=True
    )

    panels = [
        (
            frame0,
            f"Input Frame 1\n"
            f"{timestamp0:%d-%b-%Y %H:%M}"
        ),
        (
            predicted_frame,
            f"Predicted Intermediate Frame\n"
            f"{timestamp_pred:%d-%b-%Y %H:%M}"
        ),
        (
            frame1,
            f"Input Frame 2\n"
            f"{timestamp1:%d-%b-%Y %H:%M}"
        )
    ]

    for ax, (image, title) in zip(
        axes,
        panels
    ):

        ax.imshow(
            image,
            cmap="gray",
            vmin=0,
            vmax=1,
            interpolation="nearest"
        )

        ax.set_title(
            title,
            fontsize=12
        )

        ax.axis("off")

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# 16. LOAD PREVIOUSLY SAVED RESULTS
# ============================================================

if CSV_PATH.exists():

    results = pd.read_csv(
        CSV_PATH
    ).to_dict("records")

else:

    results = []


if FAILED_CSV_PATH.exists():

    failed = pd.read_csv(
        FAILED_CSV_PATH
    ).to_dict("records")

else:

    failed = []


print("\n" + "=" * 70)
print("RESUMING INSAT-3DS INTERPOLATION")
print("=" * 70)

print("Total 30-minute pairs :", len(pairs))
print("Starting pair         :", START_PAIR)
print("Previously saved      :", len(results))
print("Previously failed     :", len(failed))


# ============================================================
# 17. RESUME FROM PAIR 329
# ============================================================

start_time = time.time()

for index, (record0, record1) in enumerate(
    pairs[START_PAIR - 1:],
    start=START_PAIR
):

    timestamp0 = record0["timestamp"]

    timestamp1 = record1["timestamp"]

    timestamp_pred = (
        timestamp0 + timedelta(minutes=15)
    )

    path0 = record0["path"]

    path1 = record1["path"]

    print(
        f"\n[{index}/{len(pairs)}] "
        f"{timestamp0:%Y-%m-%d %H:%M} -> "
        f"{timestamp_pred:%Y-%m-%d %H:%M} -> "
        f"{timestamp1:%Y-%m-%d %H:%M}"
    )

    try:

        # Load original input frames
        frame0_raw = load_insat_frame(path0)

        frame1_raw = load_insat_frame(path1)

        if frame0_raw.shape != frame1_raw.shape:

            raise ValueError(
                f"Input dimensions differ: "
                f"{frame0_raw.shape} vs {frame1_raw.shape}"
            )

        # Shared normalization
        frame0, frame1, norm_low, norm_high = normalize_pair(
            frame0_raw,
            frame1_raw
        )

        # Predict intermediate 15-minute frame
        predicted_frame = predict_midpoint(
            frame0,
            frame1
        )

        time_label = timestamp_pred.strftime(
            "%Y%m%d_%H%M"
        )

        prediction_png = (
            PREDICTION_DIR
            / f"INSAT3DS_TIR1_PRED_{time_label}.png"
        )

        prediction_npy = (
            PREDICTION_DIR
            / f"INSAT3DS_TIR1_PRED_{time_label}.npy"
        )

        plot_path = (
            PLOT_DIR
            / f"INSAT3DS_15MIN_{time_label}.png"
        )

        # Save prediction as 16-bit PNG
        prediction_uint16 = np.round(
            predicted_frame * 65535
        ).astype(np.uint16)

        Image.fromarray(
            prediction_uint16
        ).save(
            prediction_png
        )

        # Save floating-point prediction
        np.save(
            prediction_npy,
            predicted_frame.astype(np.float32)
        )

        # Save three-panel plot
        save_three_panel_plot(
            frame0,
            predicted_frame,
            frame1,
            timestamp0,
            timestamp_pred,
            timestamp1,
            plot_path
        )

        # Store successful result
        results.append({

            "pair_index": index,

            "input_frame_1": str(path0),

            "input_timestamp": timestamp0.isoformat(),

            "predicted_timestamp": timestamp_pred.isoformat(),

            "input_frame_2": str(path1),

            "input_2_timestamp": timestamp1.isoformat(),

            "prediction_png": str(prediction_png),

            "prediction_npy": str(prediction_npy),

            "three_panel_plot": str(plot_path),

            "height": int(predicted_frame.shape[0]),

            "width": int(predicted_frame.shape[1]),

            "normalization_low": float(norm_low),

            "normalization_high": float(norm_high)

        })

        print(
            "Prediction saved:",
            prediction_png.name
        )

        print(
            "Three-panel plot saved:",
            plot_path.name
        )

    except Exception as exc:

        print(
            "FAILED:",
            repr(exc)
        )

        failed.append({

            "pair_index": index,

            "input_frame_1": str(path0),

            "input_frame_2": str(path1),

            "error": repr(exc)

        })

    # Save progress after every pair
    pd.DataFrame(results).to_csv(
        CSV_PATH,
        index=False
    )

    pd.DataFrame(failed).to_csv(
        FAILED_CSV_PATH,
        index=False
    )

    print(
        f"Progress saved | "
        f"Successful: {len(results)} | "
        f"Failed: {len(failed)}"
    )


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("PROCESSING FINISHED")
print("=" * 70)

print("Total available pairs :", len(pairs))

print("Started from pair     :", START_PAIR)

print("Successful predictions:", len(results))

print("Failed pairs          :", len(failed))

print(
    f"Elapsed time          : "
    f"{elapsed / 60:.2f} minutes"
)

print("\nPredicted frames :", PREDICTION_DIR)

print("Three-panel plots :", PLOT_DIR)

print("Prediction CSV   :", CSV_PATH)

print("Failed-pairs CSV :", FAILED_CSV_PATH)

if results:

    print("\nLatest saved prediction:")

    print(
        results[-1]["prediction_png"]
    )


INSAT-3DS 15-MINUTE FRAME INTERPOLATION
Device     : cuda
Data       : /home/kiruthigat.25mcse/Satellite_Frame_Interpolation/Dataset/Raw/GOES/INSAT
Model      : /home/kiruthigat.25mcse/Satellite_Frame_Interpolation/Research_RIFE/model/RIFE_CMAM_CMCL.py
Checkpoint : /home/kiruthigat.25mcse/Satellite_Frame_Interpolation/checkpoints_NEW/RIFE_CMAM_CMCL_FROM_SCRATCH/epoch_100.pth
Output     : /home/kiruthigat.25mcse/Satellite_Frame_Interpolation/results/INSAT3DS_15MIN_ONLY_THREE_PLOTS
Model wrapper: <class 'rife_cmam_cmcl.Model'>

Checkpoint type: <class 'dict'>
Checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_states', 'metrics', 'learning_rate', 'cmcl_weight', 'distill_weight']
Using checkpoint weights: model_state_dict

Selected internal module: network.flownet
Checkpoint key handling: unchanged

Checkpoint loaded successfully.
Strict loading: True
Device: cuda

Timestamped INSAT files: 426
First timestamp: 2026-07-07 00:00:00
Last timestamp : 2026-07-15 23:30:00

Total 30-minute

In [5]:

# ============================================================
# INSAT-3DS — EVALUATE SAVED 15-MINUTE INTERMEDIATE FRAMES
#
# No ground truth at T+15:
#   - Image intensity statistics
#   - MAE between T and prediction
#   - MAE between prediction and T+30
#   - MAE between prediction and linear midpoint
#   - Gradient magnitude consistency
#   - Temporal change asymmetry
#
# Outputs:
#   - CSV with per-frame diagnostics
#   - CSV with summary statistics
#
# These are indirect diagnostics, NOT ground-truth accuracy.
# ============================================================

import numpy as np
import pandas as pd
import h5py
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
from PIL import Image


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = Path(
    "/home/kiruthigat.25mcse/Satellite_Frame_Interpolation"
)

DATA_DIR = PROJECT_DIR / "Dataset" / "Raw" / "GOES" / "INSAT"

OUTPUT_DIR = (
    PROJECT_DIR / "results"
    / "INSAT3DS_15MIN_ONLY_THREE_PLOTS"
)

PREDICTION_DIR = OUTPUT_DIR / "predicted_frames"

PREDICTION_CSV = OUTPUT_DIR / "15min_predictions.csv"

METRICS_CSV = OUTPUT_DIR / "15min_indirect_evaluation.csv"
SUMMARY_CSV = OUTPUT_DIR / "15min_indirect_evaluation_summary.csv"

DATASET_NAME = "IMG_TIR1"

if not PREDICTION_CSV.exists():
    raise FileNotFoundError(
        f"Prediction CSV not found: {PREDICTION_CSV}"
    )


# ============================================================
# 2. IMAGE LOADING AND NORMALIZATION
# ============================================================

def load_raw_insat_frame(filepath):
    with h5py.File(filepath, "r") as h5:
        if DATASET_NAME not in h5:
            raise KeyError(
                f"{DATASET_NAME} not found in {filepath}"
            )

        frame = np.asarray(
            h5[DATASET_NAME][()]
        ).squeeze().astype(np.float32)

    if frame.ndim != 2:
        raise ValueError(
            f"Expected 2D image, got {frame.shape}"
        )

    return frame


def normalize_using_saved_limits(frame, low, high):
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        raise ValueError("Invalid normalization limits.")

    normalized = (frame - low) / (high - low)
    normalized = np.clip(normalized, 0.0, 1.0)
    normalized[~np.isfinite(frame)] = 0.0

    return normalized.astype(np.float32)


def load_prediction_npy(filepath):
    prediction = np.load(filepath).astype(np.float32)

    if prediction.ndim != 2:
        raise ValueError(
            f"Expected 2D prediction, got {prediction.shape}"
        )

    return prediction


# ============================================================
# 3. METRIC FUNCTIONS
# ============================================================

def mae(image_a, image_b):
    return float(np.mean(np.abs(image_a - image_b)))


def rmse(image_a, image_b):
    return float(
        np.sqrt(np.mean((image_a - image_b) ** 2))
    )


def gradient_magnitude(image):
    gy, gx = np.gradient(image)

    return np.sqrt(gx ** 2 + gy ** 2)


def image_statistics(image):
    return {
        "mean": float(np.mean(image)),
        "std": float(np.std(image)),
        "min": float(np.min(image)),
        "max": float(np.max(image))
    }


# ============================================================
# 4. EVALUATE EVERY PREDICTION
# ============================================================

prediction_df = pd.read_csv(PREDICTION_CSV)

required_columns = [
    "input_frame_1",
    "input_timestamp",
    "predicted_timestamp",
    "input_frame_2",
    "input_2_timestamp",
    "prediction_npy",
    "normalization_low",
    "normalization_high"
]

missing_columns = [
    col for col in required_columns
    if col not in prediction_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing CSV columns: {missing_columns}"
    )

results = []
failed = []

print("Predictions to evaluate:", len(prediction_df))

for idx, row in prediction_df.iterrows():
    print(
        f"\nEvaluating {idx + 1}/{len(prediction_df)}:"
        f" {row['predicted_timestamp']}"
    )

    try:
        frame0_raw = load_raw_insat_frame(
            Path(row["input_frame_1"])
        )

        frame1_raw = load_raw_insat_frame(
            Path(row["input_frame_2"])
        )

        low = float(row["normalization_low"])
        high = float(row["normalization_high"])

        frame0 = normalize_using_saved_limits(
            frame0_raw, low, high
        )

        frame1 = normalize_using_saved_limits(
            frame1_raw, low, high
        )

        predicted = load_prediction_npy(
            Path(row["prediction_npy"])
        )

        if not (
            frame0.shape == frame1.shape == predicted.shape
        ):
            raise ValueError(
                f"Image shapes differ: "
                f"T={frame0.shape}, "
                f"prediction={predicted.shape}, "
                f"T+30={frame1.shape}"
            )

        # A mathematical linear midpoint baseline.
        # This is only a comparison reference, NOT ground truth.
        linear_midpoint = 0.5 * (frame0 + frame1)

        # Differences from the real endpoint frames.
        mae_t_to_pred = mae(frame0, predicted)
        mae_pred_to_t30 = mae(predicted, frame1)

        # Difference between model output and linear midpoint.
        mae_vs_linear = mae(predicted, linear_midpoint)

        # Image gradients.
        pred_grad = gradient_magnitude(predicted)
        linear_grad = gradient_magnitude(linear_midpoint)

        gradient_difference = mae(
            pred_grad, linear_grad
        )

        # Temporal change comparison.
        change_first_half = mae(frame0, predicted)
        change_second_half = mae(predicted, frame1)

        change_asymmetry = abs(
            change_first_half - change_second_half
        )

        pred_stats = image_statistics(predicted)

        results.append({
            "input_timestamp": row["input_timestamp"],
            "predicted_timestamp": row["predicted_timestamp"],
            "input_2_timestamp": row["input_2_timestamp"],
            "mae_T_to_prediction": mae_t_to_pred,
            "mae_prediction_to_Tplus30": mae_pred_to_t30,
            "mae_prediction_vs_linear_midpoint": mae_vs_linear,
            "gradient_magnitude_difference_vs_linear": gradient_difference,
            "temporal_change_asymmetry": change_asymmetry,
            "predicted_mean_normalized": pred_stats["mean"],
            "predicted_std_normalized": pred_stats["std"],
            "predicted_min_normalized": pred_stats["min"],
            "predicted_max_normalized": pred_stats["max"]
        })

    except Exception as exc:
        print("FAILED:", repr(exc))

        failed.append({
            "predicted_timestamp": row.get(
                "predicted_timestamp", ""
            ),
            "error": repr(exc)
        })


# ============================================================
# 5. SAVE RESULTS
# ============================================================

if results:
    metrics_df = pd.DataFrame(results)
    metrics_df.to_csv(METRICS_CSV, index=False)

    metric_columns = [
        "mae_T_to_prediction",
        "mae_prediction_to_Tplus30",
        "mae_prediction_vs_linear_midpoint",
        "gradient_magnitude_difference_vs_linear",
        "temporal_change_asymmetry",
        "predicted_mean_normalized",
        "predicted_std_normalized",
        "predicted_min_normalized",
        "predicted_max_normalized"
    ]

    summary = metrics_df[metric_columns].agg(
        ["mean", "std", "min", "max"]
    ).T

    summary.to_csv(SUMMARY_CSV)

    print("\n" + "=" * 65)
    print("INDIRECT EVALUATION COMPLETED")
    print("=" * 65)
    print("Evaluated predictions:", len(metrics_df))
    print("Metrics CSV:", METRICS_CSV)
    print("Summary CSV:", SUMMARY_CSV)

    print("\nSummary:")
    print(summary.round(6))

else:
    print(
        "No predictions were evaluated successfully. "
        "Check the input paths and image shapes."
    )

if failed:
    failed_df = pd.DataFrame(failed)
    failed_path = OUTPUT_DIR / "evaluation_failed_frames.csv"
    failed_df.to_csv(failed_path, index=False)
    print("Failed evaluation records:", failed_path)


Predictions to evaluate: 420

Evaluating 1/420: 2026-07-07T00:15:00

Evaluating 2/420: 2026-07-07T00:45:00

Evaluating 3/420: 2026-07-07T01:15:00

Evaluating 4/420: 2026-07-07T01:45:00

Evaluating 5/420: 2026-07-07T02:15:00

Evaluating 6/420: 2026-07-07T02:45:00

Evaluating 7/420: 2026-07-07T04:15:00

Evaluating 8/420: 2026-07-07T04:45:00

Evaluating 9/420: 2026-07-07T05:15:00

Evaluating 10/420: 2026-07-07T05:45:00

Evaluating 11/420: 2026-07-07T06:15:00

Evaluating 12/420: 2026-07-07T06:45:00

Evaluating 13/420: 2026-07-07T07:15:00

Evaluating 14/420: 2026-07-07T07:45:00

Evaluating 15/420: 2026-07-07T08:15:00

Evaluating 16/420: 2026-07-07T08:45:00

Evaluating 17/420: 2026-07-07T09:15:00

Evaluating 18/420: 2026-07-07T09:45:00

Evaluating 19/420: 2026-07-07T10:15:00

Evaluating 20/420: 2026-07-07T10:45:00

Evaluating 21/420: 2026-07-07T11:15:00

Evaluating 22/420: 2026-07-07T11:45:00

Evaluating 23/420: 2026-07-07T12:15:00

Evaluating 24/420: 2026-07-07T12:45:00

Evaluating 25/420: 